# Qwen3-TTS single-speaker **LoRA** fine-tune (single speaker)

Same data and same pipeline as `qwen3tts_finetune_colab.ipynb`; only the training
step differs. Upstream ships no PEFT path, so step 3 injects one.

**Full FT (the other notebook):** all 905.8M params updated. 13GB peak on a T4 even
with 8-bit Adam. Each epoch checkpoint is a 3.7GB fp32 `model.safetensors`.

**LoRA (here):** base frozen, ~18M trainable adapter params (≈2%). No gradients and
no optimizer state for the frozen 905.8M, so peak drops to roughly 5GB.

**Why it may win on a small dataset:** 2% as many free parameters is 2% as much
capacity to memorise 167 clips instead of learning a voice. Full FT on 1.7 min scored 9.5% on
the timbre scale — below Kokoro — which is what overfitting sounds like.

The speaker embedding is NOT learned by gradient descent in either version: it comes
from `speaker_encoder(ref_mels)` and is written into `codec_embedding.weight[3000]`
at save time. So LoRA changes how the voice is *modelled*, not how it is *captured*.


In [ ]:
# 1. Check the GPU you actually got
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| bf16 supported:", torch.cuda.is_bf16_supported())

# --- SET THESE TO YOUR OWN ---------------------------------------------------
import os
DATASET_REPO = "your-username/your-voice-tts"   # private HF dataset repo, step 4
SPEAKER      = "myvoice"                        # name the voice registers under
os.environ["SPEAKER"] = SPEAKER                 # gen.py (step 7) reads it from env
# -----------------------------------------------------------------------------
# Left as a placeholder on purpose: this repo is public and the dataset repo id
# is not. Checked here rather than at step 4, which is a pip install, a clone
# and a patch later.
assert "your-username" not in DATASET_REPO, "set DATASET_REPO to your own repo"


In [ ]:
# 1b. Resource probe. Pure measurement -- it reads counters and changes nothing
#     about training, so numbers from a run with logging are directly comparable
#     to one without.
import subprocess, shutil, os
def res(tag=""):
    try:
        u, t = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total",
             "--format=csv,noheader,nounits"], text=True).strip().split("\n")[0].split(",")
        gpu = f"GPU {int(u)/1024:.1f}/{int(t)/1024:.1f}GB"
    except Exception:
        gpu = "GPU n/a"
    try:
        import psutil
        v = psutil.virtual_memory()
        ram = f"RAM {(v.total-v.available)/2**30:.1f}/{v.total/2**30:.1f}GB"
    except Exception:
        ram = "RAM n/a"
    d = shutil.disk_usage("/content")
    print(f"[res] {tag:24s} {gpu} | {ram} | DISK {d.used/2**30:.1f}/{d.total/2**30:.1f}GB")
res("baseline")


In [ ]:
# 2. Install + clone. Takes a few minutes.
!pip -q install -U qwen-tts accelerate
!git clone -q https://github.com/QwenLM/Qwen3-TTS.git
%cd /content/Qwen3-TTS/finetuning

# GATE 2. Everything downstream assumes this cwd and this clone. A failed pip or
# a rate-limited clone otherwise surfaces as a confusing error two cells later.
import importlib, pathlib
assert pathlib.Path.cwd() == pathlib.Path("/content/Qwen3-TTS/finetuning"), pathlib.Path.cwd()
for f in ["sft_12hz.py", "prepare_data.py", "dataset.py"]:
    assert pathlib.Path(f).exists(), f"clone incomplete: {f} missing"
import qwen_tts, accelerate, peft   # noqa: F401  -- import, not pip show: a broken
                                    # install can leave the dist-info behind
print("qwen_tts", qwen_tts.__version__ if hasattr(qwen_tts, "__version__") else "ok",
      "| accelerate", accelerate.__version__, "| peft", peft.__version__)
print("GATE 2 OK")


In [ ]:
# 3. PATCH sft_12hz.py. Five things upstream assumes that are not true here.
# Reset first: an earlier run of this cell already edited the file, so the
# replaces below would silently miss their targets. Always patch from pristine.
!git -C /content/Qwen3-TTS checkout -- finetuning/sft_12hz.py
!pip -q install -U bitsandbytes peft
# Colab preinstalls torchao 0.10.0. peft walks every dispatcher in
# lora/model.py:_create_new_module, and dispatch_torchao calls
# is_torchao_available(), which RAISES on a version below 0.16.0 instead of
# returning False (import_utils.py:147). So a library this pipeline never uses
# kills get_peft_model. Uninstalling is the fix, not upgrading: the function
# short-circuits to False when find_spec('torchao') is None, and pulling
# torchao >0.16 would drag a torch upgrade onto a working CUDA stack.
# No kernel restart needed -- sft_12hz.py runs as a subprocess and imports fresh.
!pip -q uninstall -y torchao

import torch, pathlib
p = pathlib.Path("sft_12hz.py"); s = p.read_text()

# a) flash_attention_2 needs Ampere+ (sm_80). sdpa ships with torch and works anywhere.
s = s.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')

# b) log_with="tensorboard" needs a logging_dir. init_trackers is never called and
#    nothing ever .log()s, so this is dead config -> drop it.
s = s.replace(', log_with="tensorboard"', '')

# c) T4 is Turing: no native bf16 (is_bf16_supported() says True only via emulation),
#    so upstream's bf16 has to go. PURE FP32, not fp16 autocast.
#
#    fp16 was the obvious substitute and is the wrong one. Weights must be fp32
#    regardless -- accelerator.prepare wraps the optimizer in a GradScaler, and
#    fp16 weights + a scaler raises "Attempting to unscale FP16 gradients", AMP
#    needs fp32 masters. So fp16 buys only autocast matmuls on tensor cores:
#    ~5-7 min off a 13-min run. What it costs is the whole GradScaler failure
#    mode -- on an fp16 overflow accelerate SKIPS optimizer.step() and halves the
#    scale, silently, while the step counter keeps counting -- plus the
#    instrumentation to prove that is not happening. Delete the failure class
#    instead of measuring it.
#
#    It also matches inference, which gen.py now forces to fp32 on pre-Ampere,
#    removing one axis of difference between training and generation.
#
#    MEMORY: fp16 autocast peaked at 8.8GB of 14.56GB usable. Static is ~3.9GB
#    (fp32 weights 3.7 + LoRA grads + 8-bit states), so ~4.9GB of that was
#    activations and workspace, and only that share grows here -> expect
#    ~11-13GB. Tight. If it OOMs, drop --batch_size to 1 and pass
#    --gradient_accumulation_steps 8 for the same effective batch (upstream
#    hardcodes 4 at line 44, so that needs a patch here too).
if torch.cuda.get_device_capability()[0] < 8:
    s = s.replace('mixed_precision="bf16"', 'mixed_precision="no"')
    s = s.replace("torch_dtype=torch.bfloat16", "torch_dtype=torch.float32")

# d) UPSTREAM BUG. text_embedding is nn.Embedding(vocab, text_hidden_size=2048) in
#    both sizes, but talker hidden_size is 2048 on the 1.7B and 1024 on the 0.6B.
#    The model has a text_projection MLP (modeling_qwen3_tts.py:1575) that resizes
#    2048 -> hidden_size, and every inference path uses it; sft_12hz.py:89 does not.
#    On the 1.7B that is invisible (2048->2048); on the 0.6B it is a shape error.
#    Project first, then mask -- the MLP has bias=True, so masking last is required
#    to keep padded positions at zero.
old = "input_text_embedding = model.talker.model.text_embedding(input_text_ids) * text_embedding_mask"
new = ("input_text_embedding = model.talker.text_projection(\n"
       "                    model.talker.model.text_embedding(input_text_ids)) * text_embedding_mask")
assert old in s, "line 89 not found - upstream may have fixed this"
s = s.replace(old, new)

# e) OOM. This checkpoint is 914.6M params, not 0.6B (talker.model 754.8M +
#    code_predictor 141.6M + the rest). Full fp32 AdamW = weights 3.7 + grads 3.7
#    + states 7.3 = 14.6GB, against 14.56GB usable on a T4. Batch size is
#    irrelevant -- optimizer state does not depend on it. 8-bit Adam keeps the
#    full fine-tune and drops states 7.3GB -> 1.8GB.
if torch.cuda.get_device_properties(0).total_memory < 24e9:
    s = s.replace("from torch.optim import AdamW",
                  "from bitsandbytes.optim import AdamW8bit as AdamW")

# f) LoRA. Wrap the talker (the only module that gets gradients) and train ~2% of
#    the weights. peft forwards ONE-HOP attribute access to the base module, so
#    model.talker.text_projection / .code_predictor still resolve; only
#    state_dict() key names change, handled in (g). A `.model` hop does not
#    survive the wrapper -- see (f2).
s = s.replace(
    "    config = AutoConfig.from_pretrained(MODEL_PATH)",
    "    from peft import LoraConfig, get_peft_model\n"
    "    lora = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias='none',\n"
    "                      target_modules=['q_proj','k_proj','v_proj','o_proj',\n"
    "                                      'gate_proj','up_proj','down_proj'])\n"
    "    qwen3tts.model.talker = get_peft_model(qwen3tts.model.talker, lora)\n"
    "    qwen3tts.model.talker.print_trainable_parameters()\n"
    "    config = AutoConfig.from_pretrained(MODEL_PATH)")

#    Only feed the optimizer parameters that actually require grad. Plain AdamW
#    would skip the frozen ones anyway (grad is None), but bnb's 8-bit AdamW
#    allocates per registered param, which would waste the saving.
s = s.replace(
    "optimizer = AdamW(qwen3tts.model.parameters(), lr=args.lr, weight_decay=0.01)",
    "optimizer = AdamW([p for p in qwen3tts.model.parameters() if p.requires_grad],\n"
    "                      lr=args.lr, weight_decay=0.01)")

# g) Save a MERGED full checkpoint, so inference needs no peft and the existing
#    epoch-save logic (copytree + full safetensors) works untouched.
#    merge_adapter/unmerge_adapter fold in place -- no deepcopy of 3.6GB.
#    TWO renames are needed, not one. peft registers the wrapped module as a
#    submodule (lora/layer.py:129 `self.base_layer = base_layer`), so every
#    targeted projection saves as q_proj.base_layer.weight. Stripping only the
#    talker.base_model.model. prefix leaves that infix, the base model never
#    finds q_proj.weight, from_pretrained leaves those layers at init and gen.py
#    emits noise -- after a clean run that raised nothing. The merged weights ARE
#    in base_layer.weight; only the key name is wrong.
#
#    .clone() IS LOAD-BEARING. Tensor.to('cpu') returns self when the tensor is
#    already on cpu, so without it this dict holds REFERENCES to the merged
#    weights and the unmerge_adapter() below rewinds the very values being
#    saved -- a checkpoint numerically identical to the base model, exactly the
#    failure the key-name fix above prevents, reached from the other side. On
#    cuda .to('cpu') already copies, so this is latent on Colab and fires the
#    moment training runs cpu-only, which this project has fallen back to before
#    (ft_2_cpu.wav was the only usable output of the 9.5% run). The asserts
#    below cannot catch it: the key names are right, the numbers are not.
s = s.replace(
    '            state_dict = {k: v.detach().to("cpu") for k, v in unwrapped_model.state_dict().items()}',
    "            unwrapped_model.talker.merge_adapter()\n"
    "            state_dict = {k: v.detach().to('cpu').clone()\n"
    "                          for k, v in unwrapped_model.state_dict().items()\n"
    "                          if 'lora_' not in k}\n"
    "            state_dict = {k.replace('talker.base_model.model.', 'talker.')\n"
    "                           .replace('.base_layer.', '.'): v\n"
    "                          for k, v in state_dict.items()}\n"
    "            assert not [k for k in state_dict if 'base_layer' in k or 'lora' in k], \\\n"
    "                'peft key names survived into the checkpoint'\n"
    "            assert 'talker.model.codec_embedding.weight' in state_dict, \\\n"
    "                'line 155 will KeyError'\n"
    "            unwrapped_model.talker.unmerge_adapter()")
assert "merge_adapter" in s and "get_peft_model" in s, "LoRA patches did not apply"

# g2) ATOMIC CHECKPOINT. The save is copytree(MODEL_PATH) -> rewrite config.json
#     -> save_file(model.safetensors), ~30s. Die anywhere in that window and the
#     directory is FULL SIZE, has a valid config.json, reports the speaker from
#     get_supported_speakers(), loads clean, generates clean -- and holds the
#     BASE weights, because copytree brought the base model.safetensors along and
#     save_file had not overwritten it yet. Nothing is truncated, so a size
#     heuristic cannot see it. That is a third route to a base-identical
#     checkpoint, after the .base_layer key names and the .to('cpu') aliasing,
#     and the quietest of the three.
#
#     Build in <name>.tmp and rename at the end. rename(2) within one filesystem
#     is atomic, so checkpoint-epoch-N either does not exist or is complete --
#     which also retires the >=0.9*max size filters in 7a/7b.
#
#     Ceiling: the rmtree of an existing final_dir is a ~0.1s window in which an
#     interrupt leaves a half-deleted old checkpoint. Clearing output/ before the
#     run (step 6) means it never has one to delete.
s = s.replace(
    '            output_dir = os.path.join(args.output_model_path, f"checkpoint-epoch-{epoch}")',
    '            final_dir = os.path.join(args.output_model_path, f"checkpoint-epoch-{epoch}")\n'
    '            output_dir = final_dir + ".tmp"\n'
    '            shutil.rmtree(final_dir, ignore_errors=True)\n'
    '            shutil.rmtree(output_dir, ignore_errors=True)')
s = s.replace('            save_file(state_dict, save_path)',
              '            save_file(state_dict, save_path)\n'
              '            os.rename(output_dir, final_dir)\n'
              '            accelerator.print(f"saved {final_dir}", flush=True)')
assert "os.rename(output_dir, final_dir)" in s, "(g2) did not apply"

# g3) SAVE EVERY 4th EPOCH, plus the last. Upstream saves unconditionally, which
#     was fine at 3 epochs and is not at 16: each checkpoint is ~4.4GB (3.7GB
#     safetensors plus the 682MB speech_tokenizer that copytree drags in), so 16
#     would want ~70GB against ~55GB free and the run would die on disk in the
#     back half. Every 4th gives checkpoints at epochs 3/7/11/15 -- ~17.6GB, and
#     four points to hear the voice move through rather than one endpoint.
#
#     Costs ~13 min of exposure: die before epoch 3 and there is nothing to load.
#     Acceptable against losing the run to a full disk at epoch 12.
s = s.replace(
    "        if accelerator.is_main_process:",
    "        if accelerator.is_main_process and (\n"
    "                (epoch + 1) % 4 == 0 or epoch == num_epochs - 1):")
assert "epoch == num_epochs - 1" in s, "(g3) did not apply"

# f2) PEFT SHADOWS `.model`. get_peft_model returns a PeftModel whose own .model
#     IS the LoraModel wrapper, so `model.talker.model.text_embedding` (lines 89
#     and 90) stops reaching the talker's inner model -- it lands on LoraModel and
#     raises AttributeError at the first training step, ~15 min into the run.
#     One-hop forwarding rescues .text_projection and .code_predictor (lines
#     96/111); it cannot rescue a `.model` hop. So resolve the unwrapped talker
#     once and use it for the two embedding lookups. Line 100's
#     `model.talker(...)` deliberately stays wrapped -- that is the call LoRA has
#     to intercept for anything to train at all.
anchor = "                input_codec_ids = input_ids[:, :, 1]"
assert anchor in s, "line 87 anchor not found - upstream may have moved"
s = s.replace(anchor, anchor + "\n"
              "                _tk = getattr(model.talker, 'get_base_model',\n"
              "                              lambda: model.talker)()")
s = s.replace("model.talker.text_projection(", "_tk.text_projection(")
s = s.replace("model.talker.model.text_embedding", "_tk.model.text_embedding")
s = s.replace("model.talker.model.codec_embedding", "_tk.model.codec_embedding")
assert "model.talker.model." not in s, "a .model hop survived; it will break under PEFT"
assert "outputs = model.talker(" in s, "the wrapped talker call must survive"

# h) LOGGING. Upstream prints only loss, every 10 steps, unbuffered nowhere. Two
#    things it already computes and discards:
#      - clip_grad_norm_ RETURNS the grad norm. Exploding or collapsing gradients
#        are invisible without it, and it is the honest health signal here.
#      - a non-finite loss shows up only as a weird number in a print.
#
#    NOT LOGGING GradScaler.get_scale(). Under (c) there is no scaler at all now,
#    but the number would have been misleading even under fp16: it starts at
#    65536, halves on overflow, and doubles only after growth_interval=2000
#    CONSECUTIVE good steps. This run is 4 x 84 = 336 steps, so the growth
#    interval is never reached and the scale can only fall or stay flat, by
#    construction. The early drops are the scaler calibrating -- designed
#    behaviour -- and look identical to the pathology. Alarming and correct at
#    the same time is the worst kind of metric.
#
#    `step` IS NOT A WEIGHT UPDATE. Line 71 wraps the body in
#    `with accelerator.accumulate(model)` and line 117 guards the clip with
#    `if accelerator.sync_gradients`, so with gradient_accumulation_steps=4
#    (line 44, hardcoded, no CLI arg) optimizer.step() is a no-op 3 times out of
#    4. 167 clips at batch 2 = 84 dataloader steps per epoch but only 21 real
#    updates. Upstream's counter is the dataloader one, so "Step 80" was 20
#    updates in, and reading the loss curve against it overstates the run by 4x.
#    _opt is the number that matters for whether the adapters have moved.
#
#    Both counters live inside the sync_gradients block -- that is the only place
#    a grad norm exists and the only place a step can be skipped, so they are 1:1
#    with real updates. Counting outside it would tally each norm 4 times and, on
#    non-sync steps, read a stale value from the previous sync.
s = s.replace("    model.train()",
              "    model.train()\n    _gnorm = None\n    _bad = 0\n    _opt = 0")

s = s.replace("                    accelerator.clip_grad_norm_(model.parameters(), 1.0)",
              "                    _gnorm = accelerator.clip_grad_norm_(model.parameters(), 1.0)\n"
              "                    _opt += 1\n"
              "                    if not torch.isfinite(_gnorm):\n"
              "                        _bad += 1")

_old_log = ('            if step % 10 == 0:\n'
            '                accelerator.print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")')
_new_log = (
    '            if not torch.isfinite(loss):\n'
    '                accelerator.print(f"!! NON-FINITE LOSS epoch {epoch} step {step}: {loss.item()}")\n'
    '            if step % 10 == 0:\n'
    '                import time as _t\n'
    '                _g = "n/a" if _gnorm is None else f"{float(_gnorm):.3f}"\n'
    '                accelerator.print(\n'
    '                    f"[{_t.strftime(\'%H:%M:%S\')}] Epoch {epoch} | Step {step} "\n'
    '                    f"| Loss: {loss.item():.4f} | grad_norm: {_g} "\n'
    '                    f"| updates: {_opt} | bad_grads: {_bad}",\n'
    '                    flush=True)')
assert _old_log in s, "log line not found - upstream may have changed it"
s = s.replace(_old_log, _new_log)
assert "_gnorm = accelerator.clip_grad_norm_" in s and "bad_grads" in s, "(h) did not apply"
assert "_opt += 1" in s, "(h) optimizer-step counter did not apply"

# Write LAST. This used to sit after (e), which meant (f) and (g) -- the whole
# LoRA implementation -- edited a string that was never saved: the run did a full
# fine-tune while printing that it was doing LoRA.
p.write_text(s)
assert "flash_attention_2" not in s and "log_with" not in s
assert "_tk.text_projection(" in s


print("optimizer:",
 "AdamW8bit" if "AdamW8bit" in s else "AdamW fp32")
print("sm_%d%d" % torch.cuda.get_device_capability(),
      "| mixed_precision:", "no" if 'mixed_precision="no"' in s else "bf16",
      "| weights:", "fp32" if "torch.float32" in s else "bf16")

# GATE 3. COMPILE THE RESULT. Every assert above checks that a string was
# REPLACED; none checks that what came out is still valid Python. A patch that
# lands with the wrong indentation passes all of them and then dies at step 6 --
# after a 3.7GB base download and a tokenise, ~10 min in, with a traceback
# pointing at a generated file rather than at the patch that generated it.
import py_compile, tempfile
try:
    py_compile.compile("sft_12hz.py", cfile=tempfile.mktemp(), doraise=True)
except py_compile.PyCompileError as e:
    raise SystemExit(f"PATCHED FILE DOES NOT COMPILE:\n{e}")

# And check the two structural facts the asserts above cannot see: that the
# counters landed INSIDE the sync_gradients block (outside it they would tally
# each grad norm 4x and read stale values), and that the save is still guarded.
_src = pathlib.Path("sft_12hz.py").read_text().splitlines()
_sync = next(n for n, l in enumerate(_src) if "if accelerator.sync_gradients:" in l)
_body = _src[_sync + 1:_sync + 6]
assert any("_opt += 1" in l for l in _body), "_opt is not inside the sync_gradients block"
assert any("_bad += 1" in l for l in _body), "_bad is not inside the sync_gradients block"
print("GATE 3 OK -- patched file compiles, counters are in the sync block")


In [ ]:
# 3b. sft_12hz.py does shutil.copytree(MODEL_PATH, ...) at the end of each epoch.
#     MODEL_PATH must therefore be a real local directory, not a hub id -- otherwise
#     it raises FileNotFoundError *after* the epoch has already been trained.
from huggingface_hub import snapshot_download
BASE = snapshot_download("Qwen/Qwen3-TTS-12Hz-0.6B-Base")
print(BASE)
!ls {BASE}
res("after base download")

# GATE 3b. sft_12hz.py copytrees this directory. A partial snapshot_download
# (interrupted, or a cache with a missing blob) copies fine and then produces a
# checkpoint that cannot load -- discovered at step 7, after the whole run.
import pathlib
_base = pathlib.Path(BASE)
assert (_base / "config.json").exists(), "base snapshot has no config.json"
_w = list(_base.rglob("*.safetensors")) + list(_base.rglob("*.bin"))
assert _w, "base snapshot has no weight files"
_gb = sum(f.stat().st_size for f in _base.rglob("*") if f.is_file()) / 2**30
assert _gb > 3.0, f"base snapshot is only {_gb:.1f}GB -- expected >3GB, download incomplete"
print(f"GATE 3b OK -- base is {_gb:.1f}GB, {len(_w)} weight file(s)")


In [ ]:
# 4. Pull the private dataset from HF instead of the browser upload widget.
from huggingface_hub import snapshot_download
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("HF token (hf_...): ")
snapshot_download(DATASET_REPO, repo_type="dataset",
                  local_dir=".", token=os.environ["HF_TOKEN"])
!wc -l train_raw.jsonl && ls wavs | head -3
res("after dataset pull")

# GATE 4. THE 24kHz CHECK IS THE LOAD-BEARING ONE. The 12Hz tokenizer asserts
# sr == 24000 (upstream_dataset.py:105), but a 48kHz dataset has silently reached
# this pipeline before -- and the wavs play back fine, so nothing looks wrong
# until the voice is unusable. Checked here, not after tokenising.
import json as _json, pathlib, soundfile as _sf
rows = [_json.loads(l) for l in open("train_raw.jsonl")]
assert rows, "train_raw.jsonl is empty"
_missing = [r for r in rows if not pathlib.Path(r.get("audio_path", r.get("wav", ""))).exists()]
assert not _missing, f"{len(_missing)} rows point at wavs that did not download, e.g. {_missing[0]}"

_key = "audio_path" if "audio_path" in rows[0] else "wav"
_rates, _dur = set(), 0.0
for r in rows:
    _i = _sf.info(r[_key])
    _rates.add(_i.samplerate); _dur += _i.duration
assert _rates == {24000}, f"WRONG SAMPLE RATE {_rates} -- the tokenizer needs 24000 only"
assert pathlib.Path("ref.wav").exists(), "ref.wav missing -- the speaker embedding comes from it"
print(f"GATE 4 OK -- {len(rows)} clips, {_dur/60:.1f} min, all 24kHz, ref.wav present")


In [ ]:
# 5. Extract audio codes with the 12Hz tokenizer.
!python prepare_data.py \
  --device cuda:0 \
  --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz \
  --input_jsonl train_raw.jsonl \
  --output_jsonl train_with_codes.jsonl
res("after tokenize")


In [ ]:
# 5b. Verify step 5 actually wrote codes before burning GPU time on step 6.
import json
rows = [json.loads(l) for l in open("train_with_codes.jsonl")]
# One coded row per raw row -- that is the invariant, not any fixed count. The
# 29 hardcoded here was the 1.7-min dataset; the aligned scripts give hundreds.
expected = sum(1 for _ in open("train_raw.jsonl"))
assert len(rows) == expected, f"train_raw.jsonl has {expected} rows, got {len(rows)}"
k = next(x for x in rows[0] if "code" in x.lower())
assert all(r[k] for r in rows), "some rows have empty codes"
print(len(rows), "rows | key:", k, "| first row code len:", len(rows[0][k]))


In [ ]:
!ls -la train_with_codes.jsonl && head -c 300 train_with_codes.jsonl

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# 6. LoRA fine-tune, pure fp32 (see patch (c)). Base frozen: no grads/optimizer
#    state for 905.8M params. fp16 autocast measured 8.8GB peak of 14.56GB
#    usable; fp32 grows only the activation share, so expect ~11-13GB. If it
#    OOMs, see the fallback in patch (c).
#
#    LoRA takes a HIGHER lr than full FT (1e-4 vs 2e-5) -- adapters start at zero
#    and have to travel. Full-FT LRs on the Hub are 1e-6..5e-6 for this base;
#    do not copy those here.
#
#    16 EPOCHS, AND THE REASON IS THE ACCUMULATION FACTOR. 167 clips at batch 2
#    is 84 DATALOADER steps per epoch, but sft_12hz.py:71 wraps the body in
#    `with accelerator.accumulate(model)` and gradient_accumulation_steps is 4
#    (line 44, hardcoded), so only 21 of those 84 are real weight updates.
#
#    4 epochs is therefore ~84 updates, not 336. For LoRA adapters initialised at
#    zero -- they start contributing nothing and have to travel -- that is very
#    few; hundreds is the normal range. Which means "loss flattened at 4.79" from
#    the interrupted run, at ~42 updates in, is as consistent with barely-started
#    as with converging. 16 epochs gives 336 updates and makes the curve worth
#    reading. The log now prints `updates:` alongside `Step` so the x-axis is the
#    real one.
#
#    WALL CLOCK: ~2.3s/dataloader-step measured under fp16 autocast (168 steps in
#    385s), so 16 epochs = 1344 steps = ~52 min. This run is pure fp32 (patch c),
#    which gives up the tensor-core matmuls, so expect somewhere in 60-95 min --
#    not measured, and the only number here that is a guess. Either way it is
#    nothing against a 5h30 runtime.
#
#    DISK IS NOW THE BINDING CONSTRAINT, hence patch (g3). Each checkpoint is
#    ~4.4GB (3.7GB safetensors plus the 682MB speech_tokenizer copytree drags
#    in), so saving all 16 wants ~70GB against ~55GB free -- the run would die on
#    disk around epoch 12. (g3) saves every 4th epoch plus the last: epochs
#    3/7/11/15, ~17.6GB, and four points to hear the voice move through.
#
#    The GPU sampler runs in the background because sft_12hz.py is a SUBPROCESS:
#    it releases all its memory on exit, so calling res() afterwards would just
#    report an idle card. Peak is only observable while it runs.
# Clear output/ so no stale directory survives the run. Two reasons: the last
# run left a 2.4GB checkpoint-epoch-1 that died mid-copytree and is not a
# checkpoint at all, and the atomic-save patch (g2) renames into
# checkpoint-epoch-N, which is only truly atomic when nothing is there to delete
# first.
!rm -rf output train.log gpu_train.log
res("before training")
!nohup nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits -l 5 > gpu_train.log 2>&1 &

# -u so the new per-step logs appear live, not in 4KB blocks; tee keeps a copy
# that survives a lost cell output (the first run's log only existed in the
# browser, so the interrupt nearly took the evidence with it).
!python -u sft_12hz.py \
  --init_model_path {BASE} \
  --output_model_path output \
  --train_jsonl train_with_codes.jsonl \
  --batch_size 2 \
  --lr 1e-4 \
  --num_epochs 16 \
  --speaker_name {SPEAKER} 2>&1 | tee train.log

!pkill -f "query-gpu=memory.used,utilization.gpu" || true
import subprocess
rows = [r.split(",") for r in open("gpu_train.log").read().strip().splitlines() if "," in r]
if rows:
    peak = max(int(r[0]) for r in rows)
    print(f"[res] TRAINING PEAK GPU {peak/1024:.1f}GB over {len(rows)} samples "
          f"({len(rows)*5}s), max util {max(int(r[1]) for r in rows)}%")
res("after training")
!du -sh output/checkpoint-epoch-* 2>/dev/null

# GATE 6. DID THE TRAINING ACTUALLY REACH THE CHECKPOINT? This is the one gate
# that could not exist until now, and the most important in the notebook.
#
# Three separate defects in this pipeline all produced a checkpoint that was
# byte-plausible and numerically IDENTICAL TO THE BASE MODEL: peft's
# .base_layer. key infix, .to('cpu') aliasing that unmerge_adapter then rewound,
# and the copytree window where the base model.safetensors is in place and
# save_file has not run yet. Each one loads clean, generates clean, and sounds
# like generic TTS. The asserts in patch (g) run INSIDE training and check key
# names; only comparing tensors against the base catches the values.
#
# q_proj is the probe because LoRA targeted it, so a merged save MUST have moved
# it. codec_embedding would be a false positive -- row 3000 is written
# unconditionally at sft_12hz.py:156 even by a no-op run.
import glob, pathlib
from safetensors import safe_open

cks = sorted(glob.glob("output/checkpoint-epoch-*"),
             key=lambda d: int(d.rsplit("-", 1)[1]))
assert cks, "NO CHECKPOINT AT ALL -- training did not finish an epoch"
print("checkpoints:", [c.rsplit("/", 1)[1] for c in cks])

_bw = sorted(pathlib.Path(BASE).rglob("*.safetensors"))
assert _bw, "cannot find base weights to compare against"

for ck in cks:
    _cw = pathlib.Path(ck) / "model.safetensors"
    assert _cw.exists(), f"{ck} has no model.safetensors -- save_file never ran"
    with safe_open(_cw, framework="pt") as c:
        _probe = [k for k in c.keys() if k.endswith("q_proj.weight")][:3]
        assert _probe, f"{ck} has no q_proj weights -- key renaming in (g) went wrong"
        moved = []
        for bf in _bw:
            with safe_open(bf, framework="pt") as b:
                _bk = set(b.keys())
                for k in _probe:
                    if k in _bk:
                        moved.append(not c.get_tensor(k).equal(b.get_tensor(k)))
    assert moved, f"no q_proj key of {ck} was found in the base weights -- cannot verify"
    assert any(moved), (f"{ck} IS THE BASE MODEL. {len(moved)} probed q_proj tensors are "
                        f"bit-identical to the base. Do not score this checkpoint.")
    print(f"  {ck}: {sum(moved)}/{len(moved)} probed q_proj tensors differ from base -- trained")
print("GATE 6 OK -- checkpoints carry trained weights")

# GATE 6b. The curve, against WEIGHT UPDATES rather than dataloader steps -- the
# 4x difference that made 4 epochs look like a longer run than it was. Also
# surfaces bad_grads, which should be 0 throughout in fp32: a non-zero count
# means gradients went non-finite, which is data or LR, not arithmetic.
import re
_log = pathlib.Path("train.log")
if _log.exists():
    _t = _log.read_text()
    pts = [(int(u), float(l)) for l, u in
           re.findall(r"Loss: ([\d.]+) .*?updates: (\d+)", _t)]
    if pts:
        print(f"\nloss vs updates: {pts[0][0]} updates -> {pts[0][1]:.3f} ... "
              f"{pts[-1][0]} updates -> {pts[-1][1]:.3f}")
        for u, l in pts[::max(1, len(pts) // 12)]:
            print(f"  {u:4d} updates  {l:7.3f}  {'#' * int(l * 4)}")
    _bad = [int(b) for b in re.findall(r"bad_grads: (\d+)", _t)]
    if _bad and _bad[-1]:
        print(f"\n!! bad_grads: {_bad[-1]} non-finite grad norms -- weight updates were "
              f"SKIPPED. Not expected in fp32; check the data and the LR.")
    elif _bad:
        print(f"bad_grads: 0 across {len(_bad)} log lines -- every update applied")
    if "NON-FINITE LOSS" in _t:
        print("!! NON-FINITE LOSS appeared -- grep train.log")



In [ ]:
%%writefile gen.py
# 7. Generation runs in a SUBPROCESS on purpose. A CUDA device-side assert tears
#    down the CUDA context for the whole process -- after one fires, every later
#    CUDA call in that same kernel fails, including a fresh from_pretrained on a
#    different checkpoint. In a subprocess the assert kills only the child and the
#    notebook kernel stays healthy, so no runtime restart is ever needed.
import argparse, os, time, torch, soundfile as sf
from qwen_tts import Qwen3TTSModel

TEXT = "I have been working on speech models for the last few weeks, mostly on my own laptop."

a = argparse.ArgumentParser()
a.add_argument("ckpt")
a.add_argument("--tag", default="")
a.add_argument("--greedy", action="store_true")
a.add_argument("--cpu", action="store_true")
a.add_argument("--speaker", default=os.environ.get("SPEAKER", "myvoice"))
a.add_argument("--fp32", action="store_true")
a = a.parse_args()

if a.cpu:
    dev, dt = "cpu", torch.float32
else:
    # T4 is Turing: is_bf16_supported() says True but only via emulation, which crawls.
    dev = "cuda:0"
    if a.fp32:
        dt = torch.float32   # fp16 generation failed on T4; fp32 inference is only 3.7GB
    else:
        dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("device", dev, "| dtype", dt, "| greedy", a.greedy)

t0 = time.time()
tts = Qwen3TTSModel.from_pretrained(a.ckpt, device_map=dev, dtype=dt,
                                    attn_implementation="sdpa")
print("  loaded in", round(time.time() - t0), "s")
print("  speakers:", tts.model.get_supported_speakers())

# do_sample=False ALONE IS NOT GREEDY on the 12Hz model. The sub-talker
# (code_predictor) has its own switch, subtalker_dosample, which defaults to True
# independently of do_sample -- inference/qwen3_tts_model.py:325 hard_defaults,
# picked up whenever the caller leaves it None. Its sampled ids then index
# code_predictor.get_input_embeddings()[i] at modeling_qwen3_tts.py:1683, which
# is exactly where an out-of-range index raises a device-side assert.
#
# This matters for the diagnosis, not just correctness: the first run's "greedy
# also failed" looked like proof that bad logits were NOT the cause, since argmax
# cannot emit an out-of-range index whatever the values. But the sub-talker was
# still sampling at top_k=50, temperature=0.9, in fp16. That run was never greedy
# where it mattered, so it rules nothing out.
kw = {"do_sample": False, "subtalker_dosample": False} if a.greedy else {}
t1 = time.time()
wavs, sr = tts.generate_custom_voice(text=TEXT, language="English",
                                     speaker=a.speaker, max_new_tokens=1024, **kw)
out = "ft_" + a.ckpt.rstrip("/").split("-")[-1] + a.tag + ".wav"
sf.write(out, wavs[0], sr)
print("  wrote", out, round(len(wavs[0]) / sr, 2), "s audio in",
      round(time.time() - t1), "s")
peak = torch.cuda.max_memory_allocated() / 2**30 if dev != "cpu" else 0
import resource
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20   # macOS bytes / linux KB
print(f"  peak GPU {peak:.2f}GB | peak RSS {rss:.2f}GB")


In [ ]:
# 7a. A 2x2, not a fix. The two axes were never separated: every passing run so
#     far was CPU + fp32 + sampled and every failing one GPU + fp16, so device,
#     precision and sampling all moved together.
#
#       --fp32    off/on   -> tests precision
#       --greedy  off/on   -> tests sampling  (BOTH flags, see gen.py)
#
#     THE DECISIVE CELL IS fp16 + greedy. The suspected chain is: fp16 overflows
#     -> non-finite logits -> multinomial on NaN probs (CUDA does not validate)
#     -> a garbage token id -> the embedding lookup at
#     modeling_qwen3_tts.py:1684 asserts. Remove sampling and that chain cannot
#     run, whatever the precision. So fp16+greedy passing confirms the chain end
#     to end; fp16+greedy failing means fp16 breaks something past sampling and
#     precision is the whole story. --fp32 alone would fix the symptom and
#     explain nothing.
#
#     WHY ALL FOUR OF THE FIRST RUN'S ATTEMPTS DIED, INCLUDING BOTH "GREEDY"
#     ONES: gen.py passed only do_sample=False, and subtalker_dosample defaults
#     to True independently (qwen3_tts_model.py:325/346), so the code predictor
#     sampled every time. Those runs were never greedy where it mattered and
#     ruled nothing out. gen.py now sets both.
#
#     CUDA_LAUNCH_BLOCKING=1 because a device-side assert is asynchronous:
#     without it the error surfaces at some unrelated later op with no kernel
#     name, which is exactly the generic message the first run reported. It
#     serialises every launch, so it is slow -- fine for a few short
#     generations, never leave it on for training.
#
#     Grep stderr for "Assertion" rather than tailing it: an assert prints one
#     line per offending block/thread, thousands of them, so the previous
#     stderr[-800:] would have truncated away the one line that names the fault.
import subprocess, glob, os, torch

cks = sorted(glob.glob("output/checkpoint-epoch-*"),
             key=lambda d: int(d.rsplit("-", 1)[1]))
assert cks, "no checkpoint - did step 6 finish an epoch?"
env = {**os.environ, "CUDA_LAUNCH_BLOCKING": "1"}
turing = torch.cuda.get_device_capability()[0] < 8

# The fp16 arm exists only on pre-Ampere; on sm80+ the default is bf16, which
# does not have this failure mode.
runs = []
if turing:
    runs = [("fp16_sampled", [], []), ("fp16_greedy", [], ["--greedy"])]

def go(ck, tag, flags):
    print("==", ck, tag)
    r = subprocess.run(["python", "gen.py", ck, "--tag", "_" + tag] + flags,
                       capture_output=True, text=True, env=env)
    print(r.stdout.strip())
    if r.returncode != 0:
        hits = list(dict.fromkeys(l.strip() for l in r.stderr.splitlines()
                                  if "Assertion" in l or "Error" in l))
        print("  RC", r.returncode)
        print("  " + "\n  ".join(hits[:12]) if hits else r.stderr.strip()[-800:])

for tag, prec, samp in runs:
    go(cks[-1], tag, prec + samp)

# EVERY checkpoint gets fp32 greedy AND fp32 sampled. The deliverable of this run
# is score-vs-epoch, not a final number: 336 updates on 20 min from one speaker is
# the range where LoRA starts to overfit, and (g3) saved epochs 3/7/11/15 as
# exactly the instrument for finding out. The peak may not be at the end -- if it
# lands at 11 and falls at 15 that is a real result, and scoring only the last one
# cannot tell undertrained from overtrained.
#
# GREEDY IS THE ONE TO SCORE. Sampled generation at temperature 0.9 varies run to
# run, so a sampled score-vs-epoch curve measures sampling luck alongside
# training. Greedy is deterministic: differences between epochs are the weights.
# Sampled is kept because it is what the voice actually sounds like in use.
prec = ["--fp32"] if turing else []
for ck in cks:
    go(ck, "fp32_greedy", prec + ["--greedy"])
    go(ck, "fp32_sampled", prec)

# GATE 7. A wav on disk is not a successful generation. gen.py exits 0 for
# digital silence, for a 0.2s stub, and for a full-length burst of noise -- and
# 7c plays them all through the browser widget where silence is easy to miss.
# Checked here so a dead checkpoint cannot reach timbre_score.py, which would
# happily return a percentage for it.
import glob, numpy as np, soundfile as sf
_out = sorted(glob.glob("ft_*.wav"))
assert _out, "no wav was produced by any run above"
print()
for f in _out:
    y, sr = sf.read(f)
    dur, peak = len(y) / sr, float(np.abs(y).max())
    rms = float(np.sqrt((y.astype("float64") ** 2).mean()))
    # ~85 chars of text: under 2s means it stopped early, over 20s means it ran away
    flags = []
    if peak < 1e-4:            flags.append("SILENT")
    elif rms < 1e-3:           flags.append("near-silent")
    if dur < 2.0:              flags.append("TRUNCATED")
    elif dur > 20.0:           flags.append("RAN AWAY (hit max_new_tokens?)")
    if peak > 0.999:           flags.append("clipping")
    print(f"  {f:34s} {dur:5.2f}s  peak {peak:.3f}  rms {rms:.4f}  "
          f"{'  '.join(flags) or 'ok'}")
_good = [f for f in _out if sf.info(f).duration >= 2.0]
assert _good, "every generation is silence or a stub -- nothing worth scoring"
print(f"\nGATE 7: {len(_good)}/{len(_out)} usable. Score the _fp32_greedy ones "
      f"(deterministic) for the epoch curve.")


In [ ]:
# 7b. CPU diagnosis. Only worth running if 7a's fp32 runs ALSO failed -- if fp32
#     generated cleanly, precision was the whole story and there is nothing here
#     to find. On CPU, PyTorch raises a real IndexError naming the offending
#     index instead of an async device assert.
#
#     THE OLD HYPOTHESIS HERE WAS WRONG AND IS RECORDED BECAUSE IT LOOKED RIGHT.
#     It said: the talker samples a codec_0 >= 2048 which then indexes
#     code_predictor's embeddings, whose vocab_size is 2048 while the talker's is
#     3072. The vocab numbers are real (configuration_qwen3_tts.py:189 and :373)
#     but codec_0 never touches the 2048 tables. The split is explicit at
#     modeling_qwen3_tts.py:1670 vs :1684 -- codec_0 goes to the talker's OWN
#     3072-row table, and only the code predictor's groups 1..N index the
#     2048-row ones. Same split at :1623/:1625 and :1986/:1988. The predictor's
#     lm_head is Linear(hidden, 2048) (:1167), so its own tokens are
#     structurally in range. Speaker id 3000 lands in the talker's 3072 table,
#     which is exactly where the training patch writes it (sft_12hz.py:156).
#
#     So the ONLY route to an out-of-bounds index at :1684 is multinomial
#     returning garbage from non-finite probs -- i.e. the fp16 chain 7a tests,
#     and nothing else. Which is why 7a, not this cell, is the experiment.
#
#     Pick the checkpoint rather than hardcoding one. `checkpoint-epoch-2` was
#     hardcoded here, and a run interrupted after 2 epochs has only epoch-0 and
#     epoch-1, so this cell 404'd against the Hub -- from_pretrained treats a
#     missing local dir as a repo id -- and the one diagnostic that would have
#     explained the CUDA assert never ran.
#
#     No size filter any more: patch (g2) renames into place, so a checkpoint
#     directory either does not exist or is complete. The filter it replaces had
#     a hole in both directions -- it passed a full-size directory still holding
#     base weights (copytree finished, save_file had not), and with exactly one
#     checkpoint `max` was that checkpoint itself, so a truncated dir cleared its
#     own threshold.
import glob
cks = sorted(glob.glob("output/checkpoint-epoch-*"),
             key=lambda d: int(d.rsplit("-", 1)[1]))
assert cks, "no checkpoint to diagnose"
CKPT = cks[-1]
print("diagnosing", CKPT)
!python gen.py {CKPT} --tag _cpu --cpu


In [ ]:
# 7c. Listen in the browser before scoring anything.
import glob
from IPython.display import Audio, display
for f in sorted(glob.glob("ft_*.wav")):
    print(f); display(Audio(f))


In [ ]:
# 8. Push the samples back to HF - files.download() is the same browser-session
#    widget that failed in step 4, so it will not work from a terminal client.
#    Step 4 asked for a READ token; uploading needs WRITE, so ask again here
#    rather than keeping a write token live for the whole session.
import glob, getpass
from huggingface_hub import HfApi
api = HfApi(token=getpass.getpass("HF WRITE token (hf_...): "))
for f in sorted(glob.glob("ft_*.wav")):
    api.upload_file(path_or_fileobj=f, path_in_repo="finetuned/" + f,
                    repo_id=DATASET_REPO, repo_type="dataset")
    print("pushed", f)
# Then on the Mac:
#   hf download $DATASET_REPO --repo-type dataset \
#       --include "finetuned/*" --local-dir tts_models/voice_clone/dataset/finetune_out


## Compare against the full fine-tune

Same metric, same reference (ceiling 0.9956, floor 0.8432):

| | scale |
|---|---|
| zero-shot ICL | **55.1%** |
| Kokoro (floor voice) | 12.7% |
| full FT, 1.7 min, epoch 2 | 9.5% |
| LoRA, 20.0 min (scripts 1+2) | ? |

If LoRA also lands near 10%, the dataset is the binding constraint, not the method,
the method is not the problem. Note the 9.5% run also trained on
ASR-transcribed text at ~14% WER, so it was never a clean test of full FT.
